In [1]:
import json

try:
    levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
except FileNotFoundError:
    levels = [
        {"concurrency": 1,  "tokens_per_s": 38.2,  "latency_p95_s": 0.9,  "errors": 0},
        {"concurrency": 2,  "tokens_per_s": 71.5,  "latency_p95_s": 1.1,  "errors": 0},
        {"concurrency": 4,  "tokens_per_s": 128.4, "latency_p95_s": 1.4,  "errors": 0},
        {"concurrency": 8,  "tokens_per_s": 210.7, "latency_p95_s": 2.3,  "errors": 0},
        {"concurrency": 16, "tokens_per_s": 224.9, "latency_p95_s": 5.8,  "errors": 0},
    ]
    print("using the sample bench_report -- swap in your own file for a real answer")

for L in levels:
    print(L)

{'concurrency': 1, 'tokens_per_s': 94.12, 'ttft_p50_s': 0.0505, 'ttft_p95_s': 0.0748, 'latency_p95_s': 1.3706, 'errors': 0, 'ok': 20, 'wall_s': 21.282}
{'concurrency': 2, 'tokens_per_s': 179.34, 'ttft_p50_s': 0.0511, 'ttft_p95_s': 0.0724, 'latency_p95_s': 1.406, 'errors': 0, 'ok': 20, 'wall_s': 11.169}
{'concurrency': 4, 'tokens_per_s': 304.97, 'ttft_p50_s': 0.0515, 'ttft_p95_s': 0.101, 'latency_p95_s': 1.5693, 'errors': 0, 'ok': 20, 'wall_s': 6.568}
{'concurrency': 8, 'tokens_per_s': 487.53, 'ttft_p50_s': 0.1311, 'ttft_p95_s': 0.1545, 'latency_p95_s': 1.8443, 'errors': 0, 'ok': 20, 'wall_s': 4.258}
{'concurrency': 16, 'tokens_per_s': 703.04, 'ttft_p50_s': 0.2857, 'ttft_p95_s': 0.2903, 'latency_p95_s': 2.3915, 'errors': 0, 'ok': 20, 'wall_s': 2.967}


In [2]:
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35   # a representative on-demand T4-class price; swap in your real rate

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(L["tokens_per_s"], GPU_HOURLY_USD)

for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"p95={L['latency_p95_s']:.2f}s  $/M tok=${L['cost_per_million_tokens_usd']}")

c= 1  tok/s=   94.1  p95=1.37s  $/M tok=$1.033
c= 2  tok/s=  179.3  p95=1.41s  $/M tok=$0.5421
c= 4  tok/s=  305.0  p95=1.57s  $/M tok=$0.3188
c= 8  tok/s=  487.5  p95=1.84s  $/M tok=$0.1994
c=16  tok/s=  703.0  p95=2.39s  $/M tok=$0.1383


In [3]:
TARGET_P95_S = 3.0   # your SLO from this afternoon's prediction card

under_target = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under_target, key=lambda L: L["concurrency"]) if under_target else None
print("knee:", knee)

past_knee = [L for L in levels if knee and L["concurrency"] > knee["concurrency"]]
if past_knee:
    cheapest_past_knee = min(past_knee, key=lambda L: L["cost_per_million_tokens_usd"])
    print("cheapest $/M token level past the knee (SLO-violating):", cheapest_past_knee)
    print("-> cheaper on paper, but its p95 already exceeds your SLO -- "
          "not real usable capacity at your target.")

knee: {'concurrency': 16, 'tokens_per_s': 703.04, 'ttft_p50_s': 0.2857, 'ttft_p95_s': 0.2903, 'latency_p95_s': 2.3915, 'errors': 0, 'ok': 20, 'wall_s': 2.967, 'cost_per_million_tokens_usd': 0.1383}


In [4]:
import math

def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(required_tokens_per_s, knee["tokens_per_s"])
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],   # every replica runs at the same safe knee
    }

targets = [knee["tokens_per_s"] * m for m in (1.0, 1.5, 2.0, 3.0)]
scale_plan = [scale_out_cost(t, knee, GPU_HOURLY_USD) for t in targets]
for row in scale_plan:
    print(row)

{'required_tokens_per_s': 703.04, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 2.3915}
{'required_tokens_per_s': 1054.56, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.3915}
{'required_tokens_per_s': 1406.08, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.3915}
{'required_tokens_per_s': 2109.12, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 2.3915}


In [5]:
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}
with open("cost_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "gpu_hourly_usd": 0.35,
  "target_p95_s": 3.0,
  "levels": [
    {
      "concurrency": 1,
      "tokens_per_s": 94.12,
      "ttft_p50_s": 0.0505,
      "ttft_p95_s": 0.0748,
      "latency_p95_s": 1.3706,
      "errors": 0,
      "ok": 20,
      "wall_s": 21.282,
      "cost_per_million_tokens_usd": 1.033
    },
    {
      "concurrency": 2,
      "tokens_per_s": 179.34,
      "ttft_p50_s": 0.0511,
      "ttft_p95_s": 0.0724,
      "latency_p95_s": 1.406,
      "errors": 0,
      "ok": 20,
      "wall_s": 11.169,
      "cost_per_million_tokens_usd": 0.5421
    },
    {
      "concurrency": 4,
      "tokens_per_s": 304.97,
      "ttft_p50_s": 0.0515,
      "ttft_p95_s": 0.101,
      "latency_p95_s": 1.5693,
      "errors": 0,
      "ok": 20,
      "wall_s": 6.568,
      "cost_per_million_tokens_usd": 0.3188
    },
    {
      "concurrency": 8,
      "tokens_per_s": 487.53,
      "ttft_p50_s": 0.1311,
      "ttft_p95_s": 0.1545,
      "latency_p95_s": 1.8443,
      "errors": 0,
   

In [6]:
!python /content/verify.py

recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS
